# IEEE 754: 부동소수점의 이진 내부 구조 - IEEE 754 비트 레이아웃과 정밀도 한계

- Tutorial ID: `ull-1`
- Tutorial: IEEE 754: 부동소수점의 이진 내부 구조
- Section ID: `ull-1-1`
- Section: IEEE 754 비트 레이아웃과 정밀도 한계

---

## 이 노트북에서 배우는 것

컴퓨터는 모든 데이터를 0과 1(비트)로만 저장합니다. 정수(1, 2, 3...)는 비트로 바꾸기 쉽지만,
소수(3.14, 0.1 같은 실수)는 어떻게 비트로 저장할까요?

이 질문에 대한 표준 답이 바로 **IEEE 754**입니다. 거의 모든 컴퓨터, 그리고 PyTorch/TensorFlow 같은
딥러닝 프레임워크가 이 규칙을 그대로 사용합니다. 딥러닝 모델을 FP32, FP16, BF16 등으로 학습시킬 때
"왜 정밀도가 떨어지는지", "왜 오버플로우/언더플로우가 발생하는지"를 이해하려면 IEEE 754 비트 구조를
직접 들여다보는 것이 가장 빠른 길입니다.

이 노트북은 아래 순서로 진행됩니다.

1. FP32(32비트 부동소수점) 숫자가 비트로 어떻게 쪼개지는지 직접 분해해 봅니다.
2. 10진수를 2진수로 직접 변환해보며, 왜 "근사값"이 생기는지 체감해 봅니다.
3. 부동소수점의 정밀도 한계(왜 `0.1 + 0.2 != 0.3` 인지)를 실험으로 확인합니다.
4. FP32 / FP16 / BF16 세 형식을 비교하고, 딥러닝에서 왜 BF16을 즐겨 쓰는지 알아봅니다.
5. 0, 무한대(Infinity), NaN, 비정규수(subnormal) 같은 특수한 값들의 비트 규칙을 살펴봅니다.
6. 실제 트랜스포머 모델 학습에서 이 지식이 어떻게 쓰이는지(softmax 오버플로우, 그래디언트 언더플로우) 확인합니다.

> 사전 지식: 2진수가 대략 뭔지(0과 1로 숫자를 표현한다는 정도) 알고 있으면 충분합니다.
> 모든 변환 과정은 코드 출력으로 한 단계씩 보여주므로, 외우지 않고 따라가며 이해하면 됩니다.

In [ ]:
# ============================================================
# 코드 읽는 법 — IEEE 754 비트 레이아웃과 정밀도 한계
#
# 이 노트북의 코드는 "정답을 한 번 실행해서 결과만 보는 용도"가 아니라,
# 십진수 숫자 하나가 메모리 속에서 정확히 어떤 비트 패턴으로 저장되는지를
# 한 단계씩 눈으로 확인하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 십진수 -> 비트(0과 1) -> (부호, 지수, 가수) 로 분해되는 과정을 직접 추적한다.
#   2) "정밀도 한계"가 추상적인 개념이 아니라, 실제 코드 실행 결과(예: 0.1+0.2 != 0.3)로
#      눈에 보이는 현상이라는 것을 확인한다.
#   3) FP32/FP16/BF16의 차이가 딥러닝 학습에 실제로 어떤 영향을 주는지 연결한다.
#
# 읽는 순서:
#   1) 먼저 마크다운 설명을 읽고 "이번 셀에서 무엇을 보여주려 하는지" 파악합니다.
#   2) 코드를 실행하기 전에, 출력이 어떻게 나올지 한 번 예상해 봅니다.
#   3) 실행 결과와 예상이 다르다면, 그 차이가 바로 이 노트북이 설명하려는 핵심입니다.
#   4) test_values, delta, sample 같은 변수의 숫자를 직접 바꿔서 실험해 보세요.
#
# 주의:
#   - 이 노트북은 numpy만 사용하며, 별도의 GPU나 딥러닝 프레임워크 설치가 필요 없습니다.
#   - 숫자 하나하나를 외우기보다, "왜 이런 결과가 나오는가"에 집중해서 읽어주세요.
# ============================================================

In [ ]:
import numpy as np
import struct

print("=" * 62)
print("IEEE 754 부동소수점: 비트 수준 분석")
print("=" * 62)

## 1. FP32 비트 분해

### 1-1. 32비트는 어떻게 나뉘어 있을까?

FP32(32비트 부동소수점, "single precision")는 32개의 비트를 아래처럼 3부분으로 나눠 사용합니다.

| 이름 | 비트 수 | 위치 | 역할 |
|---|---|---|---|
| 부호(Sign) | 1비트 | 맨 앞 | 0이면 양수, 1이면 음수 |
| 지수(Exponent) | 8비트 | 그 다음 | 소수점을 얼마나 이동시킬지 (2의 몇 제곱인지) |
| 가수(Mantissa, Fraction) | 23비트 | 나머지 | 실제 유효숫자(정밀도) |

이를 공식으로 쓰면 다음과 같습니다.

```
값 = (-1)^부호 × 1.가수(2진수) × 2^(지수 - 127)
```

여기서 두 가지가 낯설 수 있습니다.

**① 왜 "1.가수"일까? (암묵적 1, implicit leading 1)**
0이 아닌 숫자는 항상 "1.xxxx × 2^n" 형태로 맞출 수 있습니다. 10진수에서 과학적 표기법을 쓸 때
"항상 1~9.xxx × 10^n 형태로 쓰자"고 정한 것과 비슷합니다. 맨 앞자리를 항상 1로 고정할 수 있다면,
그 1은 따로 저장할 필요가 없으므로 비트를 하나 아낄 수 있습니다. 이것이 "암묵적 1"입니다.

**② 왜 "지수 - 127"일까? (바이어스, bias)**
지수는 음수도 될 수 있습니다(아주 작은 수를 표현할 때). 그런데 8비트로 저장된 지수 필드 자체는
0~255 사이의 "부호 없는" 정수입니다. 그래서 실제 지수값에 127을 더한 값을 저장해 두고, 나중에 꺼낼
때 다시 127을 빼서 원래 지수(-126 ~ +127)를 복원합니다. 이 127을 "바이어스(bias)"라고 부릅니다.

### 1-2. 손으로 직접 변환해보기

코드를 보기 전에, 아주 간단한 숫자 하나를 손으로 직접 변환해 봅시다. **0.5**를 예로 들겠습니다.

1. 0.5를 "1.xxx × 2^n" 형태로 바꾸면: `0.5 = 1.0 × 2^(-1)`
2. 부호: 양수이므로 `0`
3. 지수: 실제 지수가 -1이므로, 저장값 = -1 + 127 = 126 = 2진수로 `01111110`
4. 가수: 1.0은 소수부가 없으므로(이미 정확히 1.0) 가수 비트는 전부 0 → `00000000000000000000000`

따라서 0.5의 FP32 비트는 `0 01111110 00000000000000000000000` 이 되어야 합니다.
아래 코드를 실행해서 실제로 이렇게 나오는지 직접 확인해 보세요.

### 1-3. 10진수 -> 2진수 변환 과정을 한 단계씩 보기

0.5는 깔끔하게 끝나지만, 3.14처럼 애매한 소수는 2진수로 바꾸는 과정이 끝나지 않고 무한히
반복됩니다. 왜 `3.14`를 컴퓨터에 저장하면 정확히 3.14가 아니라 아주 살짝 다른 값이 되는지,
아래에서 변환 과정을 직접 출력해 봅니다.

- **정수부 변환**: 2로 나눈 나머지를 거꾸로 읽습니다. (예: 3 → "11")
- **소수부 변환**: 2를 계속 곱해서, 정수부로 튀어나오는 자리를 순서대로 읽습니다.
  (10진수에서 1/3 = 0.333... 처럼 끝나지 않는 소수가 있는 것과 같은 이유로, 2진수에서도
  끝나지 않는 소수가 있습니다. 0.1이 대표적인 예입니다.)

In [ ]:
def decimal_to_binary_steps(n, max_steps=20):
    """10진수 n을 2진수로 변환하는 과정을 한 단계씩 출력합니다.

    정수부와 소수부를 나눠서 처리합니다.
    - 정수부: '2로 나누고 나머지를 적는다'를 반복 -> 나머지를 거꾸로 읽으면 2진수가 된다.
    - 소수부: '2를 곱해서, 튀어나온 정수부가 1이면 그 자리는 1, 아니면 0'을 반복한다.
              (소수부가 정확히 0이 되지 않으면 영원히 끝나지 않을 수 있다 -> 이것이 '근사'의 원인)
    """
    integer_part = int(n)
    fraction_part = n - integer_part

    # --- 정수부 변환: 2로 나눈 나머지를 모아서 거꾸로 읽기 ---
    print(f"  [정수부 변환] {integer_part}")
    int_bits = []
    temp = integer_part
    if temp == 0:
        print("    정수부가 0이므로 그대로 0")
        int_bits = ['0']
    else:
        while temp > 0:
            bit = temp % 2  # 2로 나눈 나머지가 이번 자리의 비트
            print(f"    {temp} ÷ 2 = {temp // 2} (나머지 {bit})")
            int_bits.append(str(bit))
            temp //= 2
        int_bits.reverse()  # 나머지를 거꾸로 읽어야 올바른 2진수가 된다
    print(f"    -> 정수부 2진수: {''.join(int_bits)}")

    # --- 소수부 변환: 2를 곱해서 정수부로 튀어나오는 값을 순서대로 모으기 ---
    print(f"\n  [소수부 변환] {fraction_part:.10f}")
    frac_bits = []
    temp = fraction_part
    for step in range(max_steps):
        temp *= 2                # 소수부에 2를 곱한다
        bit = int(temp)          # 곱한 결과의 정수부가 이번 자리의 비트
        print(f"    × 2 = {temp:.10f} -> 정수부 비트 {bit}")
        temp -= bit               # 다음 단계를 위해 정수부를 떼어낸다(남은 소수부만 사용)
        frac_bits.append(str(bit))
        if temp == 0:
            break
    if temp == 0:
        ended = "(정확히 끝남)"
    else:
        ended = f"(소수부가 {max_steps}단계 후에도 안 끝남 -> 무한소수이므로 근사 필요)"
    print(f"    -> 소수부 2진수: {''.join(frac_bits)} {ended}")

    return ''.join(int_bits), ''.join(frac_bits)


print("[예시 1] 0.5는 깔끔하게 끝나는 경우")
decimal_to_binary_steps(0.5)

print("\n[예시 2] 3.14는 끝나지 않는 경우 (근사 오차의 원인)")
decimal_to_binary_steps(3.14)

### 1-4. 코드로 비트 분해하기

이제 위에서 손으로 한 과정을 함수로 만들어, 여러 숫자에 대해 한 번에 확인해 보겠습니다.

- `float_to_bits(f)`: 파이썬 float를 IEEE 754 FP32 형식의 32비트 문자열('0'/'1' 32개)로 바꿉니다.
  (`struct.pack`이 내부적으로 IEEE 754 인코딩을 해 주고, 우리는 그 결과를 비트 문자열로 펼쳐서
  눈으로 볼 수 있게 만들 뿐입니다.)
- `bits_to_components(bits)`: 32비트 문자열을 (부호, 지수, 가수)로 다시 쪼갭니다.

> 참고: 아래 `bits_to_components`는 "암묵적 1"이 있는 일반(정규화된) 숫자를 가정합니다.
> 0이나 비정규수(섹션 4에서 다룹니다)에는 그대로 적용할 수 없으므로, 이번 섹션에서는
> 0이 아닌 보통의 숫자들만 다룹니다.

In [ ]:
def float_to_bits(f):
    """파이썬 float(혹은 int)를 IEEE 754 FP32 32비트 문자열로 변환합니다.

    struct.pack('>f', f) : f를 '빅엔디안(>)' 순서의 4바이트(f, FP32)로 인코딩합니다.
                            이 4바이트 안에 이미 IEEE 754 규칙대로 부호/지수/가수가 채워져 있습니다.
    이후 각 바이트(0~255)를 8자리 이진수 문자열로 바꿔서 이어붙이면, 총 32자리 비트 문자열이 됩니다.
    """
    packed = struct.pack('>f', f)                 # f -> 4바이트(bytes)
    bits = ''.join(f'{b:08b}' for b in packed)     # 바이트마다 8자리 이진수로 바꿔서 모두 이어붙임
    return bits


def bits_to_components(bits):
    """32비트 문자열을 (부호, 지수, 가수)로 분해합니다.

    비트 위치 (왼쪽이 0번째):
      bits[0]      : 부호(sign) 1비트
      bits[1:9]    : 지수(exponent) 8비트
      bits[9:32]   : 가수(mantissa) 23비트
    """
    sign = int(bits[0])               # '0' 또는 '1' -> 정수 0/1
    exponent_bits = bits[1:9]         # 지수 8비트 (문자열 그대로 보존, 출력용)
    mantissa_bits = bits[9:32]        # 가수 23비트 (문자열 그대로 보존, 출력용)
    exponent = int(exponent_bits, 2)  # 지수 비트를 10진 정수로 변환 (아직 바이어스를 빼기 전)

    # 가수 값 계산: "1.가수" 형태이므로 1.0에서 시작해서, 비트가 1인 자리마다 2^-(자리수)를 더한다.
    # 예) 가수 비트가 "10000..." 이면 1.0 + 2^-1 = 1.5
    mantissa = 1.0
    for i, b in enumerate(mantissa_bits):
        if b == '1':
            mantissa += 2 ** -(i + 1)  # i번째 비트는 2^-(i+1) 자리값을 의미

    return sign, exponent, exponent_bits, mantissa_bits, mantissa


# 위에서 손으로 계산한 0.5를 먼저 검증해 봅니다.
print("[검증] 1-2에서 손으로 계산한 0.5와 비교")
check_bits = float_to_bits(0.5)
print(f"  계산 결과 비트: {check_bits}")
print(f"  손으로 예측한 비트: 0 01111110 00000000000000000000000 (공백 제거 시 동일해야 함)")

### 1-5. 여러 숫자에 적용해보기

이제 다양한 숫자에 똑같이 적용해 봅니다. 아래 출력을 볼 때 이런 점들을 확인해보세요.

- `1.0`과 `-1.0`은 부호 비트만 다르고 나머지는 완전히 같아야 합니다.
- `0.5`는 위 1-2에서 손으로 계산한 값과 일치해야 합니다.
- `100.0`처럼 큰 수는 지수가 더 커지고, `1e-7`처럼 작은 수는 지수가 더 작아집니다(음수).
- `1e38`은 FP32가 표현할 수 있는 한계(약 3.4×10^38)에 가까운 값입니다. 지수 비트가 거의 끝까지
  차 있는 것을 확인해보세요.

In [ ]:
test_values = [1.0, -1.0, 0.5, 3.14, 0.1, 100.0, 1e-7, 1e38]

print("\n1. FP32 비트 분해")
print("-" * 50)

for val in test_values:
    bits = float_to_bits(val)
    sign, exp, exp_bits, man_bits, mantissa = bits_to_components(bits)

    sign_str = '+' if sign == 0 else '-'
    actual_exp = exp - 127  # 저장된 지수에서 바이어스(127)를 빼면 실제 지수

    print(f"\n  {val:>12g} =")
    print(f"    비트: [{bits[0]}] [{exp_bits}] [{man_bits}]")
    print(f"          S     E(8bit)     M(23bit)")
    print(f"    분해: {sign_str}1 × 2^({exp}-127) × {mantissa:.6f}")
    print(f"         = {sign_str}{mantissa:.6f} × 2^{actual_exp}")

### 1-6. 검증: 분해한 걸 다시 합치면 원래 숫자가 나올까?

분해가 맞았는지 확인하는 가장 쉬운 방법은, (부호, 지수, 가수)를 가지고 다시 숫자를 조립해서
원래 값과 비교해 보는 것입니다. 1-1의 공식을 그대로 코드로 옮기면 됩니다.

```
값 = (-1)^부호 × 가수 × 2^(지수-127)
```

In [ ]:
print("[검증] 분해 -> 재조립 -> 원본과 비교")
for val in test_values:
    bits = float_to_bits(val)
    sign, exp, exp_bits, man_bits, mantissa = bits_to_components(bits)

    sign_mult = -1 if sign == 1 else 1
    reconstructed = sign_mult * mantissa * (2.0 ** (exp - 127))  # 공식을 그대로 코드로 옮긴 것

    match = "일치" if np.isclose(val, reconstructed, rtol=1e-6) else "불일치"
    print(f"  원본 {val:>12g}  ->  재조립 {reconstructed:>12g}   [{match}]")

## 2. 정밀도 한계

### 2-1. "정밀도가 유한하다"는 게 무슨 뜻일까?

가수(mantissa)가 23비트라는 것은, 1.0과 2.0 사이에 표현할 수 있는 숫자가 2^23(약 838만) 개뿐이라는
뜻입니다. 이 말은 두 가지를 의미합니다.

1. 1.0과 2.0 사이에는 약 838만 개의 "계단"만 존재하고, 그 사이의 값은 가장 가까운 계단으로
   반올림됩니다.
2. 지수가 커질수록(숫자가 커질수록) 같은 23비트로 더 넓은 범위를 나눠야 하므로, 계단 사이의
   간격(이를 **ULP**, Unit in the Last Place라고 부릅니다)도 커집니다. 즉, 숫자가 클수록 정밀도는
   더 거칠어집니다.

이를 십진수로 환산하면 "23비트 ≈ 약 7자리의 십진수 정밀도"가 됩니다. 즉, FP32는 대략 소수점 이하
7번째 자리부터는 신뢰할 수 없습니다.

### 2-2. 머신 엡실론(machine epsilon)

"1.0에 더했을 때 차이가 생기는 가장 작은 값"을 머신 엡실론이라고 부릅니다. 이론적으로 FP32의 머신
엡실론은 `2^-23 ≈ 1.19e-7` 입니다. 아래에서 직접 실험해 봅니다.

In [ ]:
print("\n\n2. 정밀도 한계")
print("-" * 50)

print("\n가수(23비트)가 표현할 수 있는 정밀도:")
print(f"  2^(-23) = {2**-23:.10f}")
print(f"  ≈ 1.19e-7 (약 7자리 십진수 정밀도)")

# 정밀도 한계 시연: 1.0에 아주 작은 값을 더했을 때, 그 값이 살아남는지 확인
print("\n[실험 1] 1.0 + delta - 1.0 의 결과")
a = np.float32(1.0)
for delta in [1e-6, 1e-7, 1e-8, 1e-9]:
    result = np.float32(a + delta) - a
    lost = "OK (값이 살아남음)" if result > 0 else "LOST! (반올림되어 0이 됨)"
    print(f"  1.0 + {delta:.0e} - 1.0 = {result:.2e}  [{lost}]")

print(f"\n  np.finfo(np.float32).eps = {np.finfo(np.float32).eps:.2e}  <- 이론값(1.19e-7)과 비교해 보세요")

### 2-3. 왜 `0.1 + 0.2 != 0.3` 일까?

0.1과 0.2는 둘 다 (3.14처럼) 2진수로는 정확히 표현할 수 없는 무한소수입니다. 그래서 둘 다 컴퓨터에
저장되는 순간 이미 아주 살짝 다른 값으로 근사되고, 더한 결과도 0.3의 근사값과 정확히 일치하지
않습니다. 직접 확인해 봅시다.

In [ ]:
print("\n[실험 2] 0.1 + 0.2 == 0.3 일까?")
x = 0.1 + 0.2
print(f"  0.1 + 0.2 = {x!r}")
print(f"  0.3       = {0.3!r}")
print(f"  0.1 + 0.2 == 0.3 ?  ->  {x == 0.3}")
print(f"  실제 차이: {x - 0.3:.2e}  (아주 작지만 0이 아닙니다)")

### 2-4. 큰 수 근처에서는 정밀도가 더 거칠어진다

2-1에서 말했듯, 숫자가 커지면 계단(ULP) 간격도 커집니다. 그래서 큰 수에 작은 수를 더해도
"통째로 무시"되는 경우가 생깁니다. 이런 현상을 **catastrophic cancellation(자릿수 손실)**과
연결지어 기억해두면 좋습니다.

In [ ]:
print("\n[실험 3] 큰 수 + 작은 수 = 큰 수 (작은 수가 무시됨)")
big = np.float32(1e8)
added = np.float32(big + np.float32(1.0))
print(f"  {big:.0f} + 1.0 = {added:.0f}")
print(f"  실제로 더해졌는가(added > big)? -> {added > big}")
print("  -> 1e8 부근의 ULP(계단 간격)가 이미 1보다 커서, '+1'이 반올림 과정에서 사라집니다.")

## 3. FP32 vs FP16 vs BF16 비트 비교

### 3-1. 왜 형식이 여러 개 필요할까?

지금까지 본 FP32(32비트)는 정밀도가 높지만, 숫자 하나당 4바이트를 씁니다. 딥러닝 모델은 파라미터가
수십억 개에 달하기 때문에, 메모리와 연산 속도를 아끼기 위해 더 작은 형식(16비트)을 쓰고 싶어집니다.
그런데 16비트를 "지수에 몇 비트, 가수에 몇 비트" 줄지에 따라 성격이 완전히 달라집니다.

- **FP16** (지수 5비트, 가수 10비트): 가수가 비교적 많아 정밀도는 괜찮지만, 지수가 적어서 표현
  가능한 범위가 좁습니다(너무 크거나 작은 수에서 쉽게 오버플로우/언더플로우가 일어납니다).
- **BF16** (지수 8비트, 가수 7비트): FP32와 지수 비트 수가 똑같아서 표현 범위는 FP32와 동일합니다.
  대신 가수가 적어 정밀도가 떨어집니다.

즉, "범위(지수)"와 "정밀도(가수)" 중 어느 쪽을 더 희생하느냐의 트레이드오프입니다. 딥러닝 학습에서는
그래디언트 값이 아주 크거나 아주 작아질 수 있어서, 정밀도보다 "오버플로우/언더플로우가 안 나는 것"이
더 중요한 경우가 많습니다. 그래서 최근 대형 모델 학습에는 BF16이 자주 쓰입니다.

In [ ]:
print("\n3. 포맷 비교 (비트 레이아웃)")
print("-" * 50)

# (이름, 부호 비트 수, 지수 비트 수, 가수 비트 수, 바이어스)
formats = [
    ("FP32", 1, 8, 23, 127),
    ("FP16", 1, 5, 10, 15),
    ("BF16", 1, 8, 7, 127),
]

print(f"  {'Format':>6}  {'Sign':>4}  {'Exp':>5}  {'Man':>5}  {'Bias':>5}  {'Range':>12}  {'Precision':>10}")
print("  " + "-" * 60)
for name, s, e, m, bias in formats:
    max_exp = (2**e - 1) - bias        # 표현 가능한 가장 큰 지수 (지수 비트가 전부 1이면 inf/NaN 영역이라 그 직전까지)
    min_exp = 1 - bias                 # 표현 가능한 가장 작은 "정규" 지수
    max_val = (2 - 2**-m) * 2**max_exp  # 가수가 최대일 때의 값 = 표현 가능한 가장 큰 수
    eps = 2**-m                         # 가수 1비트가 의미하는 최소 간격 = 정밀도 수준
    print(f"  {name:>6}  {s:>4}  {e:>5}  {m:>5}  {bias:>5}  2^{max_exp:<5}  {eps:.2e}")

print("""
  시각적 비교:
  FP32: [S][EEEEEEEE][MMMMMMMMMMMMMMMMMMMMMMM]   32비트 (지수 8, 가수 23)
  FP16: [S][EEEEE][MMMMMMMMMM]                   16비트 (지수 5, 가수 10)
  BF16: [S][EEEEEEEE][MMMMMMM]                   16비트 (지수 8, 가수 7)

  BF16이 FP32와 같은 지수 범위를 가지는 이유:
  -> 지수 비트가 동일! (8비트)
  -> 가수만 줄임 (23->7비트)
  -> 범위(표현 가능한 최대/최소 크기)는 유지, 정밀도(소수점 아래 자릿수)만 감소
""")

### 3-2. 직접 비교: 같은 숫자를 세 형식에 저장하면?

자연상수 e(=2.718281828...)를 FP32, FP16, BF16으로 각각 저장했을 때 오차가 얼마나 다른지 직접
비교해 봅니다.

> numpy는 bfloat16을 기본으로 제공하지 않으므로, FP32 비트의 하위 16비트를 잘라내는 방식으로
> BF16을 흉내냅니다. 실제 BF16 하드웨어는 값을 가장 가까운 BF16으로 "반올림"하는데, 여기서는
> 그냥 "잘라내기(truncate)"만 하므로 오차가 약간 더 크게 나올 수 있습니다. 그래도 "정밀도가
> 줄어드는 느낌"을 보여주는 용도로는 충분합니다.

In [ ]:
print("\n[직접 비교] 자연상수 e를 세 형식으로 저장하면?")

def to_bfloat16_approx(f):
    """FP32 비트 패턴에서 가수 하위 16비트를 잘라내어(truncate) BF16을 흉내냅니다.
    (실제 하드웨어는 반올림을 하지만, 여기서는 '정밀도가 줄어드는 느낌'을 보여주는 게 목적입니다.)
    """
    packed = struct.pack('>f', np.float32(f))
    bits_int = int.from_bytes(packed, 'big')
    bf16_bits = bits_int & 0xFFFF0000      # 하위 16비트(가수의 뒷부분)를 0으로 밀어버림
    return struct.unpack('>f', bf16_bits.to_bytes(4, 'big'))[0]

sample = np.e
fp32_val = np.float32(sample)
fp16_val = np.float16(sample)
bf16_val = to_bfloat16_approx(sample)

# 주의: 오차는 항상 float64로 변환해서 계산합니다.
# (낮은 정밀도 값과 그대로 비교하면, 비교 자체가 그 낮은 정밀도로 수행되어
#  오차가 0으로 묻혀버릴 수 있기 때문입니다.)
err_fp32 = abs(np.float64(sample) - np.float64(fp32_val))
err_fp16 = abs(np.float64(sample) - np.float64(fp16_val))
err_bf16 = abs(np.float64(sample) - np.float64(bf16_val))

print(f"  원본 (float64) : {sample:.10f}")
print(f"  FP32           : {fp32_val:.10f}   오차: {err_fp32:.2e}")
print(f"  FP16           : {fp16_val:.10f}   오차: {err_fp16:.2e}")
print(f"  BF16(근사)     : {bf16_val:.10f}   오차: {err_bf16:.2e}")
print("\n  -> FP16이 BF16보다 오차가 작습니다 (가수 비트가 더 많기 때문입니다).")
print("  -> 하지만 FP16은 지수 비트가 적어서, 아주 크거나 작은 값에서는 오히려 BF16보다 불리합니다.")

print("\n[범위 비교] 큰 수에서는 무슨 일이 벌어질까?")
big_number = 70000.0   # FP16이 표현 가능한 최댓값(약 65504)을 넘는 값
print(f"  원본: {big_number}")
with np.errstate(over='ignore'):   # 의도적으로 오버플로우를 일으키는 예제이므로 경고를 숨김
    print(f"  FP16으로 저장: {np.float16(big_number)}   <- 표현 범위를 넘어서 inf가 됨 (오버플로우)")
print(f"  BF16(근사)로 저장: {to_bfloat16_approx(big_number)}   <- 지수 범위가 FP32와 같아서 문제 없음")

## 4. 비정규수와 특수값

### 4-1. 특수한 비트 패턴들

지수 필드가 모두 0이거나 모두 1인 경우는 "일반적인 숫자"가 아니라 특별한 의미로 예약되어 있습니다.

| 지수(8비트) | 가수(23비트) | 의미 |
|---|---|---|
| `00000000` (=0) | `00000...0` | 0 (부호에 따라 +0 또는 -0) |
| `00000000` (=0) | ≠ 0 | 비정규수(subnormal) — 아래에서 설명 |
| `11111111` (=255) | `00000...0` | ±무한대(Infinity) |
| `11111111` (=255) | ≠ 0 | NaN (Not a Number, "숫자가 아님") |
| 그 외 (1~254) | 무엇이든 | 보통의(정규화된) 숫자 — 지금까지 본 모든 예시 |

### 4-2. 비정규수(subnormal)는 왜 필요할까?

정규화된 숫자는 항상 "1.xxx × 2^지수" 형태였습니다(맨 앞자리가 항상 1). 그런데 지수가 표현 가능한
가장 작은 값(-126)에 도달한 뒤에도 더 작은 수를 표현하고 싶으면 어떻게 할까요?

이때 "암묵적 1"을 포기하고 "0.xxx × 2^-126" 형태로 바꿔서, 0과 가장 작은 정규화 숫자 사이의 빈
공간을 채웁니다. 이를 **비정규수(subnormal/denormal)**라고 부르며, 정밀도는 점점 떨어지지만
(가수 앞자리에 0이 늘어날수록 유효숫자가 줄어듦) 0으로 갑자기 뚝 떨어지지 않고 "서서히" 0에
가까워지게 해줍니다(이를 **gradual underflow**라고 부릅니다).

In [ ]:
print("4. 특수값")
print("-" * 50)

special = [
    (0.0, "양의 영"),
    (-0.0, "음의 영"),
    (float('inf'), "양의 무한대"),
    (float('-inf'), "음의 무한대"),
    (float('nan'), "NaN"),
]

for val, name in special:
    try:
        bits = float_to_bits(val)
        print(f"  {name:>12}: [{bits[0]}][{bits[1:9]}][{bits[9:]}]")
    except Exception:
        print(f"  {name:>12}: (표현 불가)")

print("""
  규칙:
  - 지수=0, 가수=0 -> 0 (부호에 따라 +0, -0)
  - 지수=255, 가수=0 -> ±Infinity
  - 지수=255, 가수≠0 -> NaN
  - 지수=0, 가수≠0 -> 비정규수 (subnormal)
""")

### 4-3. 비정규수를 직접 비트로 확인해보기

FP32에서

- 가장 작은 **정규화** 양수는 `2^-126`
- 가장 작은 **비정규수**(=가장 작은 양수 전체)는 `2^-149`
- 그보다 더 작으면(`2^-150`) 더는 표현할 수 없어서 그냥 0.0이 됩니다 (언더플로우)

아래에서 직접 비트를 찍어보면, 비정규수의 지수 비트가 모두 `0`인 것을 확인할 수 있습니다.

In [ ]:
print("[비정규수 예시] 정규화된 가장 작은 수 vs 비정규수 vs 표현 불가 영역")

smallest_normal    = np.float32(2**-126)   # 정규화된 수 중 가장 작은 값 (지수가 최소, 가수=0)
subnormal_example   = np.float32(2**-130)  # 그보다 작은 값 -> 비정규수로 표현됨
smallest_subnormal  = np.float32(2**-149)  # FP32가 표현할 수 있는 가장 작은 양수(비정규수)
too_small           = np.float32(2**-150)  # 이보다 더 작으면 표현 불가 -> 0 (언더플로우)

examples = [
    (smallest_normal,   "가장 작은 정규화 수 (2^-126)"),
    (subnormal_example, "비정규수 예시 (2^-130)"),
    (smallest_subnormal, "가장 작은 비정규수 (2^-149)"),
    (too_small,          "표현 범위 밖 (2^-150)"),
]

for val, label in examples:
    bits = float_to_bits(val)
    print(f"  {label:>28}: 값={val:.3e}  비트=[{bits[0]}][{bits[1:9]}][{bits[9:]}]")

print("\n  -> 비정규수(2^-130, 2^-149)는 지수 비트가 전부 0인 것을 확인하세요(암묵적 1이 사라진 상태).")
print("  -> 2^-150은 가수/지수를 아무리 조합해도 표현이 안 돼서 그냥 0.0이 되어버립니다(언더플로우).")

## 5. 트랜스포머에서의 수치 문제

### 5-1. 왜 딥러닝에서 이게 중요할까?

트랜스포머의 attention 연산은 내부적으로 큰 점수(score) 값에 지수함수(exp)를 적용하는 softmax를
사용합니다. 그런데 `exp(1000)` 같은 값은 FP32 표현 범위(약 3.4×10^38)를 가볍게 넘어서 바로
오버플로우(→ `inf`)가 나버립니다. 그리고 일단 `inf`가 나오면 그 뒤의 거의 모든 연산이 `NaN`으로
오염되기 쉽습니다.

### 5-2. 안전한 softmax: 최댓값을 빼는 트릭

수학적으로 `softmax(x_i) = exp(x_i) / Σ exp(x_j)` 는, 모든 x에서 같은 값 c를 빼도 결과가
똑같습니다. 분자와 분모에 공통으로 `exp(-c)`가 곱해질 뿐이라, 나누는 과정에서 약분되어
사라지기 때문입니다.

```
softmax(x_i) = exp(x_i - c) / Σ exp(x_j - c)    (어떤 c를 골라도 수학적으로 결과는 동일)
```

그래서 `c = max(x)`로 고르면, 지수의 입력값(`x_i - c`)이 항상 0 이하가 되어 `exp()` 결과가 0~1
사이로만 나오고, 오버플로우가 사라집니다. 이것이 실제 딥러닝 프레임워크들이 쓰는 "안정적인
softmax"의 구현 방식입니다(흔히 log-sum-exp 트릭이라고도 부릅니다).

In [ ]:
print("5. 트랜스포머 수치 문제 사례")
print("-" * 50)

# 소프트맥스 오버플로우: attention score가 너무 크면 exp()가 바로 터진다
x = np.array([1000.0, 1001.0, 1002.0], dtype=np.float32)
print(f"  입력(예: attention score): {x}")

with np.errstate(over='ignore', invalid='ignore'):
    naive = np.exp(x)
    print(f"  exp(x) 그대로 계산: {naive}  <- 오버플로우! (FP32 표현 범위를 넘어서 inf)")

# 안전한 방법: 최댓값을 뺀 뒤 exp를 취한다 (수학적으로 결과는 동일)
x_shifted = x - np.max(x)          # 가장 큰 값을 0으로 만들고, 나머지는 음수가 되도록 이동
stable = np.exp(x_shifted)         # 이제 exp() 입력이 모두 0 이하이므로 0~1 사이 값만 나옴
result = stable / stable.sum()     # 정규화(합이 1이 되도록)
print(f"  exp(x - max)으로 계산: {np.round(stable, 4)}")
print(f"  softmax 결과:          {np.round(result, 4)}  <- 정상")

### 5-3. 그래디언트 언더플로우

이번에는 반대 방향의 문제입니다. 학습 중 그래디언트(기울기) 값은 모델이 깊어지고 학습이 진행될수록
아주 작아질 수 있습니다(이른바 vanishing gradient와 비슷한 현상입니다). 만약 이 값을 정밀도가 낮은
FP16에 저장하면, 표현 가능한 가장 작은 값보다 작은 그래디언트는 그냥 0이 되어버려서 학습 신호가
완전히 사라집니다.

In [ ]:
print("\n[그래디언트 언더플로우] 작은 값일수록 저정밀도 형식에서 사라지기 쉽다")
print(f"  FP16 최소 양수(정규화):  {np.finfo(np.float16).tiny:.2e}")
print(f"  FP32 최소 양수(정규화):  {np.finfo(np.float32).tiny:.2e}")
print(f"  학습 중 전형적 그래디언트 크기: 약 1e-4 ~ 1e-8")
print(f"  -> FP16에서는 아주 작은 그래디언트가 0으로 사라질 수 있습니다 (정보 손실)")

# 실제로 FP16에 저장했을 때 어디서부터 사라지는지 직접 확인
print("\n[실험] 다양한 크기의 그래디언트를 FP16으로 저장하면?")
grads_fp32 = np.array([1e-4, 1e-6, 1e-8, 1e-9, 1e-10], dtype=np.float32)
grads_fp16 = grads_fp32.astype(np.float16)
for g32, g16 in zip(grads_fp32, grads_fp16):
    status = "사라짐! (0이 됨)" if (g16 == 0 and g32 != 0) else "유지됨"
    print(f"  {g32:.1e} (FP32) -> {g16:.1e} (FP16)   [{status}]")

print("""
  -> 이런 이유로 실무에서는 'mixed precision' 기법을 씁니다:
     - 행렬곱 같은 무거운 연산은 빠른 FP16/BF16으로 하고,
     - 그래디언트를 누적하거나 옵티마이저가 가중치를 업데이트할 때는
       정밀도가 높은 FP32 '마스터 가중치'를 따로 유지합니다.
""")

## 정리 및 더 생각해보기

이 노트북에서 확인한 핵심 내용을 정리하면:

1. **FP32 비트 구조**: 부호(1) + 지수(8) + 가수(23) = 32비트.
   `값 = (-1)^부호 × 1.가수 × 2^(지수-127)`
2. **정밀도는 유한하다**: 가수 비트 수만큼만 정밀하고, 숫자가 커질수록 계단(ULP) 간격도 커진다.
   → `0.1 + 0.2 != 0.3`, "큰 수 + 작은 수 = 큰 수(작은 수 무시)" 같은 현상이 모두 여기서 나온다.
3. **FP32 / FP16 / BF16의 트레이드오프**: 지수 비트는 "범위", 가수 비트는 "정밀도"를 결정한다.
   BF16은 FP32와 같은 범위를 유지하면서 정밀도만 낮춰서, 큰/작은 그래디언트가 흔한 딥러닝 학습에
   적합하다.
4. **특수값과 비정규수**: 지수가 전부 0/1인 경우는 0, 무한대, NaN, 비정규수로 예약되어 있고,
   비정규수는 0으로 "뚝" 떨어지지 않고 "서서히" 줄어들게 해주는 안전장치다.
5. **딥러닝에서의 실전 적용**: softmax는 오버플로우를 피하려고 최댓값을 빼고 계산하고(log-sum-exp
   트릭), 작은 그래디언트의 언더플로우를 막기 위해 FP32 마스터 가중치를 따로 두는 mixed precision
   기법을 쓴다.

### 직접 해보면 좋은 실험

- `test_values` 리스트에 자신이 궁금한 숫자(예: `0.3`, `-3.14`, `1e-300`)를 추가해서 비트 패턴을
  확인해보세요.
- `decimal_to_binary_steps()` 함수에 `0.25`, `0.1`, `0.7` 등을 넣어보고, 어떤 수는 깔끔하게 끝나고
  어떤 수는 끝나지 않는지 비교해보세요. (힌트: 분모가 2의 거듭제곱인 분수만 깔끔하게 끝납니다.)
- `to_bfloat16_approx()` 함수에 다른 숫자를 넣어서, FP16/BF16 오차가 숫자 크기에 따라 어떻게
  달라지는지 관찰해보세요.